# SATAY-ViT End-to-End Training Notebook — Version 4 (MLCoT)
This notebook trains the **full SATAY-ViT V4 pipeline** end-to-end.

**What is new in V4 (vs V2):**
- The `task_embedding` inside `MEViTHead` is **frozen** and loaded from
  LLM-distilled knowledge vectors (`task_knowledge_vectors.pt`).
- All other components (YOLO backbone, FPN @ **16×16**, standard TransformerEncoder)
  are **identical to V2**.

**Prerequisite:** run `generate_knowledge_embeddings.py` once to create
`weights_e2e/task_knowledge_vectors.pt` before starting training.

**Training strategy:** backbone is frozen for the first N epochs (head warm-up), then unfrozen for joint fine-tuning.

In [1]:
import os, sys, torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

PIPELINE_DIR   = 'e:/DVcon/DVcon/Pipelines/satay_vit'
VERSION_DIR    = os.path.join(PIPELINE_DIR, 'Version_4')
DATA_ROOT      = 'e:/DVcon/DVcon/Data_Preprocessed'         # 16x16 dataset
WEIGHTS_DIR    = os.path.join(VERSION_DIR, 'weights_e2e')
KNOWLEDGE_PATH = os.path.join(WEIGHTS_DIR, 'task_knowledge_vectors.pt')

# --- Hyper-parameters ---
EPOCHS        = 10
BATCH_SIZE    = 16
LR            = 5e-5     # head learning rate
BACKBONE_LR   = 5e-6     # lower lr for pretrained backbone
FREEZE_EPOCHS = 3        # warm-up phase: backbone frozen for this many epochs
EMBED_DIM     = 256

if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
if VERSION_DIR not in sys.path:
    sys.path.insert(0, VERSION_DIR)

os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Verify the knowledge vectors exist before proceeding
assert os.path.exists(KNOWLEDGE_PATH), (
    f'Knowledge vectors not found at:\n  {KNOWLEDGE_PATH}\n'
    f'Run generate_knowledge_embeddings.py first!'
)
print(f'Knowledge vectors found: {KNOWLEDGE_PATH}')

print(f'CUDA:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Knowledge vectors found: e:/DVcon/DVcon/Pipelines/satay_vit\Version_4\weights_e2e\task_knowledge_vectors.pt
CUDA:   True
GPU:    NVIDIA GeForce RTX 4060 Laptop GPU
Device: cuda


## 1. Dataset (16×16 grid — same as V2)

In [2]:
from utils.data_loader import COCOTasksDataset, custom_collate

# grid_size=16 matches V4's FPNAggregator.GRID
train_ds = COCOTasksDataset(DATA_ROOT, split='train', grid_size=16)
val_ds   = COCOTasksDataset(DATA_ROOT, split='test',  grid_size=16)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          persistent_workers=True,
                          num_workers=4, pin_memory=True, collate_fn=custom_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          persistent_workers=True,
                          num_workers=4, pin_memory=True, collate_fn=custom_collate)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Loaded 100800 PREPROCESSED samples for train split. Target grid: 16x16
Loaded 12600 PREPROCESSED samples for test split. Target grid: 16x16
Train batches: 6300 | Val batches: 788


## 2. Model (V4 — Frozen MLCoT Knowledge Embedding)

The only change vs V2: `MEViTHead.task_embedding` is loaded from the
pre-computed **MLCoT knowledge vectors** and is **frozen** (not trained).
All other weights (YOLO backbone, FPN, Transformer layers, q/k projections)
are trainable as before.

In [3]:
from Version_4.model_e2e import SATAYViT_E2E

model = SATAYViT_E2E(
    embed_dim      = EMBED_DIM,
    knowledge_path = KNOWLEDGE_PATH,
).to(device)

trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_pars = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters : {trainable:,}  (frozen task_embedding excluded)')
print(f'Total parameters     : {total_pars:,}')

Trainable parameters : 1,383,680  (frozen task_embedding excluded)
Total parameters     : 2,338,656


## 3. Optimizer, Scheduler & Loss

In [4]:
criterion = nn.BCELoss()

# Separate learning rates: backbone gets 10x lower lr.
# The frozen task_embedding parameters (requires_grad=False) are
# automatically excluded from the optimizer.
backbone_params = list(model.backbone.parameters())
head_params = [
    p for p in
    list(model.aggregator.parameters()) + list(model.vit_head.parameters())
    if p.requires_grad   # exclude frozen task_embedding
]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params,     'lr': LR},
], weight_decay=1e-4)

# Cosine annealing: smoothly decays lr to 5% of initial over all epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.05)
print('Optimizer and scheduler ready.')

Optimizer and scheduler ready.


## 4. End-to-End Training Loop

In [5]:
import json
import os
import torch

def set_backbone_grad(requires_grad):
    for p in model.backbone.parameters():
        p.requires_grad = requires_grad

best_val_loss    = float('inf')
train_history, val_history = [], []
ckpt_path   = os.path.join(WEIGHTS_DIR, 'satay_vit_e2e_best.pt')

START_EPOCH = 0
RESUME_CHECKPOINT = os.path.join(WEIGHTS_DIR, 'satay_vit_e2e_latest.pt')

# Resume training from a specific epoch if a latest checkpoint exists
if os.path.exists(RESUME_CHECKPOINT):
    print(f'Resuming from checkpoint {RESUME_CHECKPOINT}')
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)
    START_EPOCH = checkpoint['epoch']
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))

    # Restore training history if it exists
    hist_path = os.path.join(WEIGHTS_DIR, 'training_history.json')
    if os.path.exists(hist_path):
        with open(hist_path, 'r') as f:
            hist = json.load(f)
            train_history, val_history = hist.get('train', []), hist.get('val', [])

# Initial freeze logic on resume
if START_EPOCH < FREEZE_EPOCHS:
    set_backbone_grad(False)
    print(f'Phase 1: warming up head for {FREEZE_EPOCHS - START_EPOCH} epochs (backbone frozen)...')
else:
    set_backbone_grad(True)
    print(f'Phase 2: backbone active — fine-tuning end-to-end...')

for epoch in range(START_EPOCH, EPOCHS):
    if epoch == FREEZE_EPOCHS and START_EPOCH < FREEZE_EPOCHS:
        set_backbone_grad(True)
        print('\nPhase 2: backbone unfrozen — fine-tuning end-to-end...')

    # ── Train ──────────────────────────────────────
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for batch in pbar:
        imgs     = batch['image'].to(device)
        tasks    = batch['task_id'].to(device)
        gt_hmaps = batch['heatmap'].to(device)

        optimizer.zero_grad()
        pred = model(imgs, tasks)
        loss = criterion(pred, gt_hmaps)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train = train_loss / len(train_loader)
    scheduler.step()

    # ── Validate ───────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Valid]'):
            imgs     = batch['image'].to(device)
            tasks    = batch['task_id'].to(device)
            gt_hmaps = batch['heatmap'].to(device)
            val_loss += criterion(model(imgs, tasks), gt_hmaps).item()

    avg_val = val_loss / len(val_loader)
    train_history.append(avg_train)
    val_history.append(avg_val)
    phase = 'frozen' if epoch < FREEZE_EPOCHS else 'active'
    print(f'  Epoch {epoch+1:>2} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | backbone={phase}')

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'epoch':          epoch + 1,
            'state_dict':     model.state_dict(),
            'optimizer':      optimizer.state_dict(),
            'best_val_loss':  best_val_loss,
            'knowledge_path': KNOWLEDGE_PATH,   # store for evaluation
        }, ckpt_path)
        print(f'  ==> Saved best checkpoint -> {ckpt_path}')

    # Save epoch/latest checkpoints for resume support
    latest_ckpt = os.path.join(WEIGHTS_DIR, 'satay_vit_e2e_latest.pt')
    epoch_ckpt  = os.path.join(WEIGHTS_DIR, f'satay_vit_e2e_epoch_{epoch+1}.pt')
    ckpt_dict = {
        'epoch':          epoch + 1,
        'state_dict':     model.state_dict(),
        'optimizer':      optimizer.state_dict(),
        'best_val_loss':  best_val_loss,
        'knowledge_path': KNOWLEDGE_PATH,
    }
    torch.save(ckpt_dict, latest_ckpt)
    torch.save(ckpt_dict, epoch_ckpt)

    # Save training history
    with open(os.path.join(WEIGHTS_DIR, 'training_history.json'), 'w') as f:
        json.dump({'train': train_history, 'val': val_history}, f)

print(f'\nBest Val Loss: {best_val_loss:.4f}')

Phase 1: warming up head for 3 epochs (backbone frozen)...


Epoch 1/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

Epoch 1/10 [Valid]:   0%|          | 0/788 [00:00<?, ?it/s]

  Epoch  1 | Train: 0.1374 | Val: 0.1155 | backbone=frozen
  ==> Saved best checkpoint -> e:/DVcon/DVcon/Pipelines/satay_vit\Version_4\weights_e2e\satay_vit_e2e_best.pt


Epoch 2/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

...

Epoch 7/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

Epoch 7/10 [Valid]:   0%|          | 0/788 [00:00<?, ?it/s]

  Epoch  7 | Train: 0.0760 | Val: 0.1121 | backbone=active


Epoch 8/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Loss Curve

In [ ]:
from utils.plot_metrics import plot_training_losses
from IPython.display import Image, display

plot_path = plot_training_losses(
    train_history, val_history,
    save_dir=WEIGHTS_DIR
)
display(Image(plot_path))

## 6. Evaluation: Top-1 Task-Aware Accuracy
Runs YOLO inference + ME-ViT relevance scoring + score fusion.
Uses the **best** checkpoint saved during training.

In [ ]:
from utils.evaluate_map_variants import evaluate_best_model

evaluate_best_model(
    data_root    = DATA_ROOT,
    weights_path = ckpt_path,
    output_dir   = VERSION_DIR,
    use_original_data = False,
    iou_thresholds = [0.5],
)